# Test Engine

File: test_engine.py

In [ ]:
import sys
import os


In [ ]:
import unittest
import numpy as np
from game.connect4_env import ConnectFourEnv


In [ ]:
"""
Tests tự động cho game engine và các agents
=============================================
Chạy: python tests/test_engine.py
      python -m pytest tests/ -v
"""

import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))


In [ ]:
class TestConnectFourEnv(unittest.TestCase):
    """Kiểm thử cơ bản của game engine."""

    def setUp(self):
        self.env = ConnectFourEnv()
        self.env.reset()

    # ----- Initial state -----

    def test_initial_state_zeros(self):
        """Bàn cờ ban đầu phải toàn số 0."""
        self.assertTrue(np.all(self.env.state == 0))

    def test_initial_turn_red(self):
        """Đỏ đi trước."""
        self.assertEqual(self.env.turn, 'red')

    def test_initial_valid_inputs(self):
        """7 cột đều hợp lệ ban đầu."""
        self.assertEqual(sorted(self.env.validinputs), list(range(1, 8)))

    def test_initial_not_done(self):
        """Game chưa kết thúc."""
        self.assertFalse(self.env.done)

    # ----- Action space -----

    def test_action_space_n(self):
        self.assertEqual(self.env.action_space.n, 7)

    def test_observation_space_shape(self):
        self.assertEqual(self.env.observation_space.shape, (7, 6))

    # ----- Step -----

    def test_step_returns_correct_shape(self):
        state, reward, done, info = self.env.step(4)
        self.assertEqual(state.shape, (7, 6))

    def test_step_places_piece_in_correct_column(self):
        self.env.step(1)  # red plays column 1
        self.assertEqual(self.env.state[0][0], 1)  # bottom of col 0 (index)

    def test_step_alternates_turns(self):
        self.env.step(1)
        self.assertEqual(self.env.turn, 'yellow')
        self.env.step(2)
        self.assertEqual(self.env.turn, 'red')

    def test_step_pieces_stack(self):
        """Quân xếp chồng từ dưới lên."""
        self.env.step(4)  # red
        self.env.step(4)  # yellow
        self.assertEqual(self.env.state[3][0], 1)   # red at row 0
        self.assertEqual(self.env.state[3][1], -1)  # yellow at row 1

    # ----- Win detection -----

    def test_win_vertical(self):
        """4 quân đỏ xếp dọc → thắng."""
        for _ in range(4):
            self.env.step(4)  # red
            if not self.env.done:
                self.env.step(1)  # yellow plays elsewhere
        self.assertTrue(self.env.done)
        self.assertEqual(self.env.reward, 1)

    def test_win_horizontal(self):
        """4 quân đỏ xếp ngang → thắng."""
        for col in [1, 2, 3]:
            self.env.step(col)   # red
            self.env.step(col)   # yellow (cùng cột)
        self.env.step(4)  # red plays 4th horizontal
        self.assertTrue(self.env.done)
        self.assertEqual(self.env.reward, 1)

    def test_invalid_action_raises(self):
        """Hành động không hợp lệ → ValueError."""
        with self.assertRaises(ValueError):
            self.env.step(8)

    def test_full_column_removed_from_valid(self):
        """Cột đầy phải bị xóa khỏi validinputs."""
        for _ in range(3):
            self.env.step(1)   # red
            self.env.step(1)   # yellow
        self.assertNotIn(1, self.env.validinputs)

    def test_sample_returns_valid_action(self):
        """sample() trả về hành động hợp lệ."""
        for _ in range(20):
            action = self.env.sample()
            self.assertIn(action, self.env.validinputs)

    def test_copy_independence(self):
        """copy() tạo bản sao độc lập."""
        self.env.step(4)
        copy = self.env.copy()
        copy.step(3)
        # Original không bị thay đổi
        self.assertEqual(self.env.state[2][0], 0)

    def test_reset_clears_board(self):
        """reset() xóa bàn cờ."""
        self.env.step(4)
        self.env.reset()
        self.assertTrue(np.all(self.env.state == 0))
        self.assertEqual(self.env.turn, 'red')


In [ ]:
class TestAgents(unittest.TestCase):
    """Kiểm thử các agents cơ bản."""

    def setUp(self):
        self.env = ConnectFourEnv()
        self.env.reset()

    def test_random_agent_valid(self):
        from agents.random_agent import conn_random
        for _ in range(10):
            env = ConnectFourEnv()
            env.reset()
            action = conn_random(env)
            self.assertIn(action, env.validinputs)

    def test_think1_takes_win(self):
        """Think1 phải thắng ngay nếu có thể."""
        from agents.heuristic_agent import conn_think1
        # Tạo tình huống: đỏ có 3 quân dọc ở cột 4
        env = ConnectFourEnv()
        env.reset()
        for _ in range(3):
            env.step(4)   # red
            env.step(1)   # yellow (elsewhere)
        # Đỏ có thể thắng bằng cột 4
        self.assertIn(4, env.validinputs)
        action = conn_think1(env)
        self.assertEqual(action, 4, "Think1 phải chọn cột 4 để thắng")

    def test_think2_blocks_opponent(self):
        """Think2 phải chặn đối thủ thắng ngay."""
        from agents.heuristic_agent import conn_think2
        # Tạo tình huống: vàng sắp thắng ở cột 3
        env = ConnectFourEnv()
        env.reset()
        env.step(3)  # red col 3
        env.step(1)  # yellow col 1
        env.step(5)  # red col 5
        env.step(2)  # yellow col 2
        env.step(6)  # red col 6
        # yellow có 3 quân ngang (1,2,3) - cần chặn cột 3 hoặc cột có thể thắng
        # Think2 nên chặn
        action = conn_think2(env)
        self.assertIn(action, env.validinputs)

    def test_alphabeta_returns_valid(self):
        """AlphaBeta trả về hành động hợp lệ."""
        from agents.alphabeta_agent import alphabeta_agent
        env = ConnectFourEnv()
        env.reset()
        action = alphabeta_agent(env, depth=3)
        self.assertIn(action, env.validinputs)

    def test_mcts_returns_valid(self):
        """MCTS trả về hành động hợp lệ."""
        from agents.mcts_agent import mcts_agent
        env = ConnectFourEnv()
        env.reset()
        action = mcts_agent(env, n_simulations=50)
        self.assertIn(action, env.validinputs)

    def test_full_game_random_vs_random(self):
        """Random vs Random chạy xong mà không lỗi."""
        from game.game_utils import one_conn_game
        from agents.random_agent import conn_random
        for _ in range(5):
            reward, moves = one_conn_game(conn_random, conn_random)
            self.assertIn(reward, [-1, 0, 1])
            self.assertGreater(len(moves), 0)


In [ ]:
class TestDraw(unittest.TestCase):
    """Kiểm thử trường hợp hòa."""

    def test_draw_detection(self):
        """Bàn cờ đầy mà không có người thắng → hòa."""
        env = ConnectFourEnv()
        env.reset()
        # Chơi theo thứ tự tạo hòa (không có 4 liên tiếp)
        # Pattern: RYRYRYR, YRYRYRYR, ... (7 cột, 6 hàng = 42 ô)
        # Đơn giản: để tự động test, dùng random mà không check winner
        import random
        random.seed(99)
        env2 = ConnectFourEnv()
        env2.reset()
        # Force fill board theo thứ tự không tạo win
        # Cách đơn giản: fill từng cột luân phiên
        cols_order = [1,2,3,4,5,6,7] * 6
        for col in cols_order:
            if col in env2.validinputs and not env2.done:
                env2.step(col)
        # Game phải kết thúc (dù win hay draw)
        self.assertTrue(env2.done)


In [ ]:
if __name__ == '__main__':
    unittest.main(verbosity=2)
